# 🩺 Nidaan — TrOCR Fine-Tuning on Real RxHandBD Dataset

**Before running:**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. Upload `RxHandBD.zip` using the Files panel (left sidebar → upload icon)
3. `Runtime` → `Run all`

**Expected time:** ~1.5 hours on T4 GPU
**Expected CER:** < 12% on real Bangladeshi prescription handwriting

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────
import subprocess, sys

pkgs = ['evaluate', 'jiwer', 'albumentations']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)

import torch
assert torch.cuda.is_available(), '❌ GPU not found! Go to Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'✅ PyTorch: {torch.__version__}')
import transformers
print(f'✅ Transformers: {transformers.__version__}')

In [ ]:
# ── CELL 2: Unzip RxHandBD dataset ──────────────────────────────────────
import os, zipfile

ZIP_PATH = '/content/RxHandBD.zip'
EXTRACT_DIR = '/content/RxHandBD'

assert os.path.exists(ZIP_PATH), (
    '❌ RxHandBD.zip not found!\n'
    'Upload it via the Files panel (left sidebar → upload icon).'
)

print('📦 Extracting RxHandBD.zip...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

# Verify structure
train_dir  = os.path.join(EXTRACT_DIR, 'Train_Set')
test_dir   = os.path.join(EXTRACT_DIR, 'Test_Set')
train_csv  = os.path.join(EXTRACT_DIR, 'Train_Label.csv')
test_csv   = os.path.join(EXTRACT_DIR, 'Test_Labels.csv')

n_train = len(os.listdir(train_dir))
n_test  = len(os.listdir(test_dir))

print(f'✅ Train images : {n_train}')
print(f'✅ Test  images : {n_test}')
print(f'✅ Total        : {n_train + n_test}')

In [ ]:
# ── CELL 3: Load & augment dataset ──────────────────────────────────────
import pandas as pd
import numpy as np
from PIL import Image
import albumentations as A
from datasets import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Load CSVs
train_df = pd.read_csv(train_csv)   # columns: Images, Text
test_df  = pd.read_csv(test_csv)

# Build full paths
train_df['path'] = train_df['Images'].apply(lambda f: os.path.join(train_dir, f))
test_df['path']  = test_df['Images'].apply(lambda f: os.path.join(test_dir,  f))

# Filter out any rows whose image file doesn't exist
train_df = train_df[train_df['path'].apply(os.path.exists)].reset_index(drop=True)
test_df  = test_df[test_df['path'].apply(os.path.exists)].reset_index(drop=True)

print(f'Train samples after filtering: {len(train_df)}')
print(f'Test  samples after filtering: {len(test_df)}')
print(f'\nSample labels:')
print(train_df[['Images','Text']].head(8).to_string(index=False))

# Albumentations augmentation pipeline
aug = A.Compose([
    A.Rotate(limit=5, p=0.5),
    A.GaussNoise(var_limit=(5, 25), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.ElasticTransform(alpha=0.5, sigma=5, p=0.3),
    A.Blur(blur_limit=3, p=0.2),
])

print('\n✅ Dataset loaded and augmentation pipeline ready')

In [ ]:
# ── CELL 4: Load TrOCR model ─────────────────────────────────────────────
print('🔄 Loading microsoft/trocr-small-handwritten...')

MODEL_ID = 'microsoft/trocr-small-handwritten'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# Move to GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'✅ Model loaded on {device.upper()}')

# Count parameters
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ Model parameters: {total_params:.1f}M')

In [ ]:
# ── CELL 5: Preprocess dataset ───────────────────────────────────────────
import evaluate as ev

MAX_LABEL_LEN = 32  # longest BD drug name: ~25 chars

def load_and_augment(path, is_train=True):
    img = Image.open(path).convert('RGB')
    # Resize to TrOCR expected 384x384 square
    img = img.resize((384, 64), Image.LANCZOS)
    if is_train:
        arr = np.array(img)
        arr = aug(image=arr)['image']
        img = Image.fromarray(arr)
    return img

def preprocess(batch, is_train=True):
    images = [load_and_augment(p, is_train) for p in batch['path']]
    px = processor(images=images, return_tensors='pt').pixel_values
    enc = processor.tokenizer(
        list(batch['Text']),
        padding='max_length',
        max_length=MAX_LABEL_LEN,
        truncation=True
    ).input_ids
    labels = [
        [(t if t != processor.tokenizer.pad_token_id else -100) for t in row]
        for row in enc
    ]
    return {'pixel_values': px, 'labels': labels}

print('🔄 Preprocessing training set...')
train_ds = Dataset.from_pandas(train_df).map(
    lambda b: preprocess(b, is_train=True),
    batched=True, batch_size=16, desc='Train'
)
print('🔄 Preprocessing test set...')
eval_ds = Dataset.from_pandas(test_df).map(
    lambda b: preprocess(b, is_train=False),
    batched=True, batch_size=16, desc='Eval'
)
print(f'✅ Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# ── CELL 6: Train ────────────────────────────────────────────────────────
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
)

cer_metric = ev.load('cer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    print(f'\n📊 Sample predictions:')
    for p, l in zip(pred_str[:5], label_str[:5]):
        print(f'  Pred: "{p}"  |  True: "{l}"  {"✅" if p.strip()==l.strip() else ""}')
    return {'CER': round(cer, 4)}

args = Seq2SeqTrainingArguments(
    output_dir                = './ckpts',
    num_train_epochs          = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    fp16                      = True,
    learning_rate             = 4e-5,
    warmup_steps              = 200,
    weight_decay              = 0.01,
    eval_strategy             = 'epoch',
    save_strategy             = 'epoch',
    save_total_limit          = 2,
    load_best_model_at_end    = True,
    metric_for_best_model     = 'CER',
    greater_is_better         = False,
    predict_with_generate     = True,
    generation_max_length     = MAX_LABEL_LEN,
    logging_steps             = 50,
    report_to                 = 'none',
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    data_collator   = default_data_collator,
    compute_metrics = compute_metrics,
)

print('⚡ Training on real RxHandBD data (~1.5 hours on T4)...')
print('   You can leave this tab open. Results print after each epoch.\n')
trainer.train()

trainer.save_model('./nidaan-trocr')
processor.save_pretrained('./nidaan-trocr')
print('\n🎉 Training complete! Model saved.')

In [ ]:
# ── CELL 7: Final evaluation ─────────────────────────────────────────────
print('📊 Running final evaluation on test set...')
results = trainer.evaluate()
cer_score = results.get('eval_CER', results.get('eval_cer', 'N/A'))
print(f'\n✅ Final CER on real RxHandBD test set: {cer_score}')
if isinstance(cer_score, float):
    pct = (1 - cer_score) * 100
    print(f'   Character accuracy: {pct:.1f}%')
    if cer_score < 0.12:
        print('   🎉 Excellent! Model is production-ready.')
    elif cer_score < 0.20:
        print('   ✅ Good. The fuzzy matcher will cover remaining errors.')
    else:
        print('   ⚠ Consider training more epochs or adding synthetic data.')

In [ ]:
# ── CELL 8: Export to ONNX for your HP EliteBook ─────────────────────────
print('📦 Exporting to ONNX (optimized for Intel CPU)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optimum[onnxruntime]'], check=True)

from optimum.onnxruntime import ORTModelForVision2Seq

onnx_model = ORTModelForVision2Seq.from_pretrained('./nidaan-trocr', export=True)
onnx_model.save_pretrained('./nidaan-trocr-onnx')
processor.save_pretrained('./nidaan-trocr-onnx')

import shutil
shutil.make_archive('nidaan-trocr-onnx', 'zip', '.', 'nidaan-trocr-onnx')
size_mb = os.path.getsize('nidaan-trocr-onnx.zip') / (1024*1024)
print(f'✅ ONNX package ready: nidaan-trocr-onnx.zip ({size_mb:.0f} MB)')

In [ ]:
# ── CELL 9: Download to your laptop ──────────────────────────────────────
from google.colab import files
print('⬇️ Starting download...')
files.download('nidaan-trocr-onnx.zip')
print('Done! Extract to D:\\Nidaan\\models\\ on your laptop.')